# 00-Setup: 10분 사전 점검

이 노트북은 수업 전에 끝내 두는 10분 점검입니다. 01~06은 고객 요청 하나("결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.")를 여러 구조로 처리하면서 구조를 고른 이유를 찾는 실습입니다. 그 전에 내 컴퓨터에서 네 층이 각자 돌아가는지 눈으로 확인합니다.

1. 모델을 부르지 않는 LangGraph 그래프
2. 실제 모델 호출
3. 도구 하나를 가진 Deep Agent 한 턴
4. EXA 웹 검색

## 학습 목표

- 설치는 저장소 루트에서 한 번, 실행은 `Day03_new/`에서 한다는 구분을 말합니다.
- 이 과정에 필요한 두 키(`ANTHROPIC_API_KEY`, `EXA_API_KEY`)와 각각의 비용을 내는 주체를 말합니다.
- 모델 없는 그래프, 실제 모델, Deep Agent, EXA 네 층이 각각 내 환경에서 도는지 출력으로 확인하고, 실패했을 때 어느 층의 문제인지 가려 냅니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 설치와 폴더 구조 | 2 |
| 2 | 두 키 상태와 `.env` | 2 |
| 3 | 모델 없는 그래프와 사례 데이터 | 2 |
| 4 | 실제 모델, Deep Agent, EXA | 3 |
| 5 | 이후 시간표와 비용 | 1 |

## 호출 규모

모델을 부르는 셀은 둘입니다. 모델 직접 호출 1회와 Deep Agent 한 턴입니다. Deep Agent는 도구를 쓰면 모델이 한 번 더 불리므로 두 셀을 합쳐 모델 호출은 약 3회이고, 셀마다 예고하는 상한은 합쳐 4회입니다. EXA 검색은 1회입니다. 앞쪽 그래프와 데이터 셀은 모델 호출 0회, EXA 호출 0회입니다. 비용은 내 Anthropic 크레딧과 내 EXA 크레딧에서 나갑니다. 키가 없으면 절 2의 키 확인 셀에서 한국어 안내와 함께 멈추므로 유료 호출까지 가지 않습니다.

## 1. 설치와 폴더 구조 (2분)

설치는 **저장소 루트에서 한 번만** 합니다. 루트 `pyproject.toml`이 uv workspace이고 `uv.lock`과 `.venv`는 루트에 하나만 있습니다. `Day03_new/`에는 `pyproject.toml`이 없습니다. `uv run`이 위 폴더로 올라가 루트 환경을 찾습니다.

```bash
uv sync --locked            # 저장소 루트에서 한 번만
cd Day03_new
uv run jupyter lab          # 실행은 Day03_new 폴더에서
```

`Day03_new/` 안에서 `uv sync`를 실행하지 않습니다. 이 폴더에 필요 없는 패키지가 환경에서 지워져 다른 Day 실습이 깨집니다. 잘못 실행했다면 루트에서 `uv sync --locked`로 복구합니다.

실행 위치가 중요한 이유는 하나입니다. 노트북이 `rag_lab/`, `mas_lab/` 폴더 옆에 있고 kernel의 작업 폴더가 `Day03_new/`이기 때문에 `import mas_lab`이 됩니다. 아래 두 셀은 이 kernel이 루트 `.venv`에서 도는지, 두 폴더와 실습 데이터가 제자리에 있는지 확인합니다. 모델도 EXA도 부르지 않습니다.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

print("Python            :", sys.version.split()[0])
for package in ("deepagents", "langgraph", "langchain-anthropic"):
    print(f"{package:<18}:", version(package))
print("sys.prefix        :", sys.prefix)
print("루트 .venv에서 실행 중인가:", sys.prefix.endswith(".venv"))
print("kernel 작업 폴더  :", Path.cwd().name)

In [ ]:
import importlib
from collections import Counter

from mas_lab.view import show_table
from rag_lab.corpus import load_cases, load_corpus
from rag_lab.settings import ROOT

documents, cases = load_corpus(), load_cases()
modules = sorted(p.stem for p in (ROOT / "mas_lab").glob("*.py") if p.stem != "__init__")
broken = []
for name in modules:
    try:
        importlib.import_module(f"mas_lab.{name}")
    except Exception as error:
        broken.append(f"{name}: {type(error).__name__}")
facts = {name: (ROOT / "data" / "support" / name).exists() for name in ("billing.json", "outage.json")}
show_table([
    {"항목": "ROOT 폴더", "값": ROOT.name},
    {"항목": "원문 문서", "값": f"{len(documents)}개, 고객(entity)별 " + ", ".join(f"{k} {v}" for k, v in Counter(d.entity for d in documents.values()).items())},
    {"항목": "평가 사례 수", "값": f"{len(cases)}개"},
    {"항목": "사례 유형", "값": ", ".join(f"{kind} {count}" for kind, count in Counter(c["type"] for c in cases).items())},
    {"항목": "mas_lab 모듈", "값": f"{len(modules)}개 중 import 실패 {len(broken)}개 {broken}"},
    {"항목": "고정 사실 파일", "값": str(facts)},
])

## 2. 두 키 상태와 `.env` (2분)

이 과정은 키 두 개를 씁니다.

| 키 | 쓰는 곳 | 발급 | 비용을 내는 주체 |
|---|---|---|---|
| `ANTHROPIC_API_KEY` | 모든 노트북의 모델 호출 | Anthropic 콘솔 | 내 Anthropic 크레딧 |
| `EXA_API_KEY` | 06의 필수 리서치 실습, 이 노트북의 EXA 확인 | `https://exa.ai/` (구글 로그인을 추천합니다) | 내 EXA 크레딧 |

두 키는 `Day03_new/.env`에만 둡니다. `Day03_new/.env.example`을 복사해 직접 채웁니다.

```bash
cp Day03_new/.env.example Day03_new/.env   # 그리고 편집기로 두 키를 직접 채웁니다
```

`Day02_new/.env`를 이미 갖고 있다면 그 파일을 복사한 뒤 `EXA_API_KEY`만 더하면 됩니다. `ANTHROPIC_API_KEY`와 모델 이름이 그대로 들어 있습니다. 변수 이름이 `DAY02_MODEL`, `DAY02_EFFORT`로 남아 있는 것은 그 복사본을 고치지 않고 쓰게 하려는 것입니다.

- 키 값은 노트북, 로그, 결과 파일 어디에도 출력하지 않습니다. 도구는 `설정됨`과 `미설정`만 말합니다.
- 셸에 export한 키나 다른 Day의 `.env`는 읽지 않습니다. 모르는 사이에 내가 아닌 계정의 크레딧이 쓰이는 사고를 막기 위해서입니다.
- `.env`를 고친 뒤에는 **kernel을 재시작**해야 반영됩니다.
- 이 과정은 실제 모델로만 돕니다. 키가 없으면 대체 경로로 넘어가지 않고 키 확인 셀에서 멈춥니다.

아래 첫 셀은 상태 표를 낸 뒤 두 키를 차례로 요구하고, 모델 객체를 만듭니다. 객체를 만드는 것은 호출이 아닙니다.

In [ ]:
from mas_lab.view import show_table
from rag_lab.settings import key_status, make_model, require_exa_key, require_key

show_table([{"항목": name, "상태": status} for name, status in key_status().items()])
settings = require_key()  # ANTHROPIC_API_KEY가 없으면 여기서 RuntimeError로 멈춥니다
require_exa_key()         # EXA_API_KEY가 없으면 여기서 멈춥니다
model = make_model(settings)
print("모델 객체를 만들었습니다(호출은 하지 않았습니다):", model.model, "| effort:", settings.effort)

In [ ]:
# 실패 재현: 키가 빈 설정으로 두 확인 함수를 불러 멈출 때의 안내를 본다
from rag_lab.settings import Settings

empty = Settings(api_key="")  # 메모리 안에서만 만든 빈 설정입니다. .env 파일은 건드리지 않습니다
for check in (require_key, require_exa_key):
    try:
        check(empty)
    except RuntimeError as error:
        print(f"{check.__name__:<16} -> {type(error).__name__}: {error}")

**관찰할 것.** 상태 표에서 `ANTHROPIC_API_KEY`와 `EXA_API_KEY`가 모두 `설정됨`인지 봅니다. 하나라도 `미설정`이면 위 키 확인 셀이 이미 멈췄을 것이고, 멈춘 줄이 `require_key()`인지 `require_exa_key()`인지가 어느 키를 채워야 하는지 알려 줍니다. 실패 재현 셀의 두 줄에서는 세 가지를 읽습니다. 예외 종류가 `RuntimeError`인지, 안내문이 어느 파일과 어느 변수를 가리키는지, 안내문 어디에도 키 값이 없는지입니다. 이 문구가 나오면 아직 비용은 한 번도 나가지 않은 상태입니다.

## 3. 모델 없는 그래프와 사례 데이터 (2분)

이 과정의 많은 셀은 모델을 부르지 않습니다. 그래프의 구조, reducer, 게이트, 멱등, 실패 경로는 모델의 판단이 필요 없는 것들이라 일반 함수 노드로 돌립니다. 키 없는 대체 경로가 아닙니다. 결정적인 구조를 결정적으로 보이려는 것이고, 그런 셀은 출력에 "모델 호출 0회"를 적습니다. 반대로 모델을 부르는 셀은 `announce`가 먼저 규모를 알립니다.

가장 작은 예로 병렬 그래프를 하나 돌립니다. 노드 이름 `billing`, `policy`, `outage`는 01~06의 관통 사례에서 서로 독립적으로 읽는 세 근거(결제 사실, 규정 검색, 가용률 조회)를 가리킵니다. 여기서는 각 노드가 0.3초 기다릴 뿐입니다. `Send`가 노드를 몇 개든 한 단계에서 동시에 띄우고, `Trace`가 각 노드가 일한 구간을 기록해 `overlap_seconds`가 동시에 일한 시간을 셉니다.

**먼저 예측하세요.** 같은 그래프를 두 번 돌립니다. 첫 번째는 기본 설정이고 두 번째는 `config={"max_concurrency": 1}`입니다. 각각 `전체 시간(초)`이 세 노드 대기 시간의 합과 가장 긴 노드 하나의 대기 시간 중 어느 쪽에 가까울지, `겹친 시간(초)`이 0일지 아닐지 실행 전에 적어 보세요.

In [ ]:
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from mas_lab.trace import Trace, overlap_seconds, timeline
from mas_lab.view import show_graph


class FanState(TypedDict):
    names: list[str]
    done: Annotated[list[str], operator.add]  # 동시에 쓴 값을 reducer가 이어 붙입니다


def build_fan(trace: Trace):
    def work(state: dict) -> dict:
        with trace.span(state["name"]):
            time.sleep(0.3)  # 일하는 시간을 대신하는 대기입니다. 모델 호출이 아닙니다
        return {"done": [state["name"]]}

    builder = StateGraph(FanState)
    builder.add_node("work", work)
    builder.add_conditional_edges(START, lambda s: [Send("work", {"name": n}) for n in s["names"]], ["work"])
    builder.add_edge("work", END)
    return builder.compile()


show_graph(build_fan(Trace("정의 확인")))

In [ ]:
modes = {"기본": {}, "max_concurrency=1": {"max_concurrency": 1}}
rows, spans = [], {}
for label, config in modes.items():
    trace = Trace(label)
    started = time.perf_counter()
    result = build_fan(trace).invoke({"names": ["billing", "policy", "outage"]}, config=config)
    rows.append({"실행": label, "전체 시간(초)": round(time.perf_counter() - started, 3),
                 "겹친 시간(초)": overlap_seconds(trace.spans), "끝난 노드": len(result["done"])})
    spans[label] = trace.spans
show_table(rows)
for label, items in spans.items():
    print(f"[{label}]")
    print(timeline(items))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
import json

from mas_lab.desk import load_billing, load_outage
from mas_lab.view import show_table
from rag_lab.settings import ROOT

print("안내:", json.loads((ROOT / "data" / "support" / "billing.json").read_text(encoding="utf-8"))["_note"])
show_table([{"결제 번호": p["payment_id"], "청구서": p["invoice_id"], "승인 시각": p["approved_at"], "금액": p["amount"],
             "기본 이용료분": p["base_fee_part"], "초과분": p["usage_fee_part"]} for p in load_billing("알파")])
show_table([{"월": month, "가용률(%)": fact["availability_percent"],
             "사건": ", ".join(i["incident_id"] for i in fact["incidents"]) or "없음"}
            for month, fact in load_outage("알파").items()])
print("없는 고객을 조회하면:", load_billing("없는회사"), load_outage("없는회사"))

In [ ]:
import time

from rag_lab.retrieval import tokenize

started = time.perf_counter()
tokens = tokenize("서비스 크레딧 신청 기한")
first = time.perf_counter() - started
started = time.perf_counter()
tokenize("서비스 크레딧 신청 기한")
again = time.perf_counter() - started
print("토큰:", tokens)
print(f"첫 호출 {first:.2f}초 / 같은 문장 다시 {again:.4f}초")

**관찰할 것.** 지금까지 모델 호출과 EXA 호출은 한 번도 없었으므로 비용은 0입니다. 다음 셀부터 비용이 생깁니다. 네 개의 출력을 아래 칸으로 읽습니다.

- **그래프 정의 셀**의 간선 표에서 `조건부` 칸이 `True`인 간선을 봅니다. `Send`가 몇 개로 갈라질지는 실행할 때 정해지므로 정의된 그래프에는 노드 하나만 보입니다. 병렬이 일어났는지는 그림이 아니라 아래 실행 표와 구간 기록으로 확인합니다.
- **병렬 그래프 표**의 `겹친 시간(초)` 칸입니다. 기본 행이 0보다 크고 `max_concurrency=1` 행이 0이면 병렬이 실제로 일어난 것이고, 바꾼 것은 설정 하나뿐입니다. 두 행 모두 0이면 이 환경에서 동시 실행이 일어나지 않았다는 뜻입니다. `전체 시간(초)`를 예측과 비교하고, 문자 막대에서 세로로 겹치는 구간이 있는지도 봅니다. `끝난 노드`가 두 행에서 같은 것은 병렬 여부가 결과 개수를 바꾸지 않는다는 뜻입니다. 바뀌는 것은 시간입니다.
- **결제 표**에서 `청구서` 칸 값이 같은 행을 찾아 봅니다. 같은 청구서에 승인이 둘 있는 경우가 어느 청구서인지, 두 행의 `승인 시각`이 얼마나 가까운지 읽습니다. 가용률 표에서는 `사건` 칸이 `없음`인 달과 아닌 달을 가릅니다. 마지막 줄의 빈 값은 사실이 없을 때 오류가 아니라 빈 결과가 나온다는 것을 보여 주고, 이 차이가 02 이후 "조회되지 않음"을 답으로 말해야 하는 사례와 이어집니다.
- **Kiwi 줄**에서 첫 호출과 다시 부른 시간의 차이를 봅니다. 두 값이 비슷하면 앞 셀이 이미 사전을 적재한 것입니다. 첫 호출이 눈에 띄게 길면 그 차이가 적재 시간입니다. 이 값은 컴퓨터마다 다르므로 내 출력에서만 읽습니다.

## 4. 실제 모델, Deep Agent, EXA (3분)

세 셀이 각각 다른 층을 확인합니다. 모두 실행 직전에 `announce(...)`로 규모를 한 줄 알립니다. 한 줄에 든 숫자는 그 셀의 예상 상한이고, 실제 호출 수는 아래 출력에서 다시 셉니다.

1. 모델 직접 호출: 키가 유효한지, 모델이 응답하는지, 응답이 어떤 모양인지 봅니다.
2. Deep Agent 한 턴: `create_deep_agent`에 결제 조회 도구 하나를 줍니다. 도구의 고객(entity)과 기준일(as_of)은 `Desk`가 고정하므로 모델은 인자로 고객을 고르지 못합니다. 대화가 메시지 몇 줄로 이루어지는지 표로 읽습니다.
3. EXA 검색: 웹 검색 한 번이 제목과 URL로 돌아오는지 봅니다. 본문은 출력하지 않습니다.

**먼저 예측하세요.** Deep Agent 셀을 실행하기 전에 메시지 표가 몇 줄일지, `종류` 칸이 어떤 순서(Human, AI, Tool 중)로 나올지, 모델 호출 집계가 몇 회일지 적어 보세요. 도구를 부르면 모델은 최소 몇 번 불리는지도 함께 적습니다.

In [ ]:
from rag_lab.settings import announce

announce("모델 직접 호출 확인", model_calls=1)
reply = model.invoke("한 단어로 답하세요: 준비")
usage = reply.usage_metadata or {}
print("모델 응답            :", reply.text)
print("내용 블록 종류       :", [b["type"] if isinstance(b, dict) else "text" for b in reply.content] if isinstance(reply.content, list) else "text")
print("요청한 모델 / 응답한 모델:", settings.model, "/", reply.response_metadata.get("model"))
print("입력 토큰 / 출력 토큰:", usage.get("input_tokens"), "/", usage.get("output_tokens"))

In [ ]:
from deepagents import create_deep_agent

from mas_lab.runner import make_desk
from mas_lab.trace import ModelMeter
from mas_lab.view import message_table
from rag_lab.corpus import load_cases

case = next(c for c in load_cases() if c["id"] == "complex-double-pay")
desk, meter = make_desk(case), ModelMeter()  # 고객과 기준일은 desk 안에 고정되어 있습니다
announce("Deep Agent 한 턴", model_calls=3)
agent = create_deep_agent(model=model, tools=desk.tools("billing"), system_prompt="결제 사실은 도구로 조회한 값만 말하세요.")
question = "결제 내역을 조회해서 건수와, 같은 청구서에 겹친 결제가 있는지 한 줄로 알려 주세요."
result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"callbacks": [meter]})
show_table(message_table(result["messages"]))
print("모델 호출 집계:", meter.totals())
print("조회로 등록된 사실 번호:", sorted(desk.fact_refs))

In [ ]:
from mas_lab.exa import ExaClient

announce("EXA 검색 확인", exa_calls=1)
client = ExaClient(max_searches=1, num_results=3, max_chars=300)
found = client.search("LangGraph Send API")
print(f"결과 {len(found)}건 | 검색 {client.searches}회 | {client.seconds:.2f}초 | 비용(달러) {client.cost_dollars}")
for row in found:
    print(row["source_id"], "|", row["title"][:50], "|", row["url"])

**관찰할 것.** 세 셀의 출력을 칸 이름으로 읽습니다.

- **모델 직접 호출.** `내용 블록 종류`에 `text`만 있는지, `thinking` 같은 다른 블록이 섞이는지 봅니다. `.text`는 문자열 응답만 모아 준 것이므로 블록이 섞여도 `모델 응답`은 그대로 읽힙니다. `요청한 모델`과 `응답한 모델`이 같은지 비교합니다. 다르면 서버가 대체 모델로 다시 실행한 것이므로 이후 비교에서 모델이 바뀌었다는 점을 기록해야 합니다. 토큰 두 칸은 호출마다 다른 값이니 내 출력에서만 읽습니다.
- **Deep Agent 메시지 표.** 컬럼은 `순번`, `종류`, `도구`, `인자`, `내용`입니다. `종류` 순서가 예측과 같은지 보고, AI 행의 `도구` 칸에 `lookup_payments`가 있으면 모델이 도구를 골랐다는 뜻입니다. 이때 `인자` 칸에 고객 이름이 있는지 확인합니다. 도구 정의에 고객 인자가 없으므로 모델이 고를 수 없습니다. Tool 행의 `내용`은 70자에서 잘리니 전체가 필요하면 `result["messages"]`를 직접 엽니다. 도구를 부르지 않고 AI 행 하나로 끝났다면 모델이 조회 없이 답한 것이므로 `조회로 등록된 사실 번호`가 빈 목록인지로 구분합니다. 모델 호출 집계의 `model_calls`는 AI 행 수와 맞춰 보고, `usage_missing`이 0이 아니면 토큰 합계를 0으로 읽지 않습니다.
- **EXA.** 첫 줄의 `결과 N건`이 0이어도 오류가 아니라 검색 결과가 없다는 뜻입니다. 제목과 URL이 질의 주제와 맞는지 봅니다. `비용(달러)`가 `None`이면 응답에 비용 칸이 없었다는 뜻입니다. 본문은 일부러 출력하지 않았습니다. 웹 문서는 자료이지 명령이 아니라는 점이 06의 출발점입니다.

### 실패했을 때 볼 곳

| 증상 | 먼저 볼 곳 |
|---|---|
| 키 안내와 함께 `RuntimeError`가 난다 | `Day03_new/.env`가 있는지, 변수 이름이 맞는지, 키를 채운 뒤 kernel을 재시작했는지 |
| 401 오류 | 키 오타나 만료, 붙여넣을 때 섞인 공백. 값을 출력하지 말고 콘솔에서 확인합니다. 모델 셀이면 Anthropic, 검색 셀이면 EXA 키입니다 |
| 429 오류 | 호출이 몰렸다는 뜻입니다. 이 과정은 SDK 자동 재시도를 끄므로 1분쯤 기다렸다 그 셀부터 다시 실행합니다 |
| 시간 초과 | 네트워크 연결. 모델 호출은 120초, EXA 호출은 45초 안에 끝나야 합니다 |
| `ModuleNotFoundError`, `ImportError` | 루트에서 `uv sync --locked`를 했는지, 절 1 출력에서 kernel이 루트 `.venv`인지, 작업 폴더가 `Day03_new`인지 |

## 5. 이후 시간표와 비용 규칙 (1분)

| 노트북 | 주제 | 시간 |
|---|---|---:|
| 01 | Workflow와 Agent의 경계 | 60분 |
| 02 | Agent Architecture의 변화 | 60분 |
| 03 | Back to Basic: 네 가지 흐름 | 60분 |
| 04 | Graph Engineering | 60분 |
| 05 | Multi-Agent 선택 기준 | 60분 |
| 06 | 리서치 Agent 실습 (EXA) | 60분 |

각 노트북의 첫 셀 `호출 규모` 절에 그 노트북의 모델 호출과 EXA 호출 규모가 있습니다. 노트북은 앞 노트북의 kernel 상태에 기대지 않으므로 하나만 따로 열어도 돌아갑니다.

비용 규칙은 다섯 가지입니다.

1. 유료 호출이 있는 셀은 호출 직전에 `announce(목적, model_calls=N, exa_calls=M)`로 규모를 한 줄 알립니다. `N`은 그 셀의 예상 상한입니다. Agent가 도구를 더 부르면 실제 호출은 더 나올 수 있으니 출력의 집계를 봅니다.
2. 모델 판단이 필요 없는 셀은 모델을 부르지 않고 출력에 호출 0회를 적습니다.
3. `직접 해보기`의 유료 셀은 첫 줄이 `RUN_EXERCISE = False`로 꺼져 있습니다. 코드를 고친 뒤 `True`로 바꿔야 돌고, 전체 실행에 비용을 더하지 않습니다. 대용량 선택 셀은 `RUN_OPTIONAL`로 같은 방식입니다.
4. 한 번 실행한 결과는 "1회 관찰"입니다. 같은 조건을 3회 이상 반복하기 전에는 평균이나 우열을 말하지 않습니다.
5. 비용은 내 크레딧에서 나갑니다. 키 값은 화면, 로그, 결과 파일 어디에도 남기지 않습니다.

## 직접 해보기

### 해보기 1

`Day03_new/.env`에서 `EXA_API_KEY=` 뒤의 값을 잠시 지웁니다(줄은 남겨 둡니다). kernel을 재시작하고 처음부터 끝까지 실행해, 어느 셀에서 어떤 안내와 함께 멈추는지 적어 보세요. 실행 전에 두 가지를 예측합니다. 멈추는 셀은 어디이고, 그 셀 뒤의 모델 호출 셀은 실행될까요? 실행해 본 뒤 그 사이에 비용이 나갔는지 이유와 함께 적고, 키를 되돌립니다. 이 과정에서 키 값을 화면에 띄우거나 노트북에 붙여 넣지 않습니다.

### 해보기 2

작업 폴더가 `Day03_new/`가 아니면 `import mas_lab`이 어떻게 되는지 봅니다. 아래 셀은 다른 폴더에서 새 Python을 띄워 같은 import를 시도합니다. 모델은 부르지 않으므로 그대로 실행됩니다. 먼저 실행해 종료 코드와 마지막 줄을 읽고, `RUN_FOLDER`를 `str(ROOT)`로 바꿔 다시 실행해 무엇이 달라지는지 비교한 뒤 이유를 한 줄로 적습니다. 마지막으로 터미널에서 `Day03_new/` 밖에서 `jupyter lab`을 띄우면 첫 코드 셀들이 어디서 실패할지 적어 보세요.

In [ ]:
import os
import subprocess
import sys
import tempfile

from rag_lab.settings import ROOT

RUN_FOLDER = tempfile.gettempdir()  # str(ROOT)로 바꿔 다시 실행해 보세요
no_color = {**os.environ, "PYTHON_COLORS": "0"}  # 오류 문구에 색 제어 문자가 섞이지 않게 합니다
done = subprocess.run([sys.executable, "-c", "import mas_lab"], cwd=RUN_FOLDER, capture_output=True, text=True, env=no_color)
print("작업 폴더 :", RUN_FOLDER)
print("종료 코드 :", done.returncode)
print("마지막 줄 :", (done.stderr.strip().splitlines() or ["(오류 없음)"])[-1])

## 정리

- 설치는 루트에서 `uv sync --locked` 한 번, 실행은 `Day03_new/`에서 합니다. `Day03_new/`에서 `uv sync`를 하지 않습니다.
- 키는 `Day03_new/.env`에 직접 채웁니다. `ANTHROPIC_API_KEY`는 모델, `EXA_API_KEY`는 웹 검색에 쓰이고 비용은 각자의 크레딧에서 나갑니다. 출력은 `설정됨`과 `미설정`뿐입니다.
- 이 과정은 실제 모델로만 돕니다. 모델 판단이 필요 없는 그래프 셀은 호출 0회로 결정적으로 돌리고, 유료 셀은 `announce`가 먼저 규모를 알립니다.
- 네 층(모델 없는 그래프, 모델, Deep Agent, EXA)이 각각 따로 확인되었습니다. 실패하면 어느 층인지부터 가립니다.
- 다음: `01-Workflow-vs-Agent`에서 다음 행동을 누가 정하는지부터 시작합니다.

## 확인 질문

1. 병렬 그래프 셀과 Deep Agent 셀이 각각 증명하는 것과 증명하지 않는 것은 무엇인가요?
2. 셸에 키를 export해도 `key_status()`가 `미설정`이라고 나오는 이유와, 이 설계가 막으려는 사고는 무엇인가요?